# Clasificación de Spam vs. Ham

Proyecto de clasificación de texto listo para **Google Colab**.

**Pregunta:** ¿podemos detectar si un mensaje es spam o ham?

**Target:** spam (1) vs ham (0)

**Stack:** pandas, seaborn, scikit-learn, shap

Dataset de ejemplo de mensajes.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("OK")

## 1. Cargar / simular datos de mensajes

In [ ]:
np.random.seed(RANDOM_STATE)
# Mensajes simulados simples
ham = ["Hola, cómo estás?", "Nos vemos mañana en la oficina", "Gracias por el mensaje", "Te llamo más tarde", "Ok, perfecto"]
spam = ["Ganaste un premio! Haz clic aquí", "Oferta limitada, compra ahora", "Dinero gratis, responde YA", "Tu cuenta será bloqueada", "Haz clic para reclamar tu premio"]

mensajes = []
labels = []
for _ in range(300):
    if np.random.rand() > 0.4:
        mensajes.append(np.random.choice(ham) + " " + np.random.choice(["", "Saludos", "Chau"]))
        labels.append(0)
    else:
        mensajes.append(np.random.choice(spam) + " " + np.random.choice(["", "URGENTE", "GRATIS"]))
        labels.append(1)

df = pd.DataFrame({"mensaje" : mensajes, "spam" : labels})
print(df.shape)
df.head()

## 2. EDA

In [ ]:
print(df["spam"].value_counts())
print(df["spam"].value_counts(normalize=True).round(3))

In [ ]:
sns.countplot(data=df, x="spam")
plt.title("Distribución de mensajes")
plt.xticks([0, 1], ["Ham", "Spam"])
plt.show()

### Hallazgos
- Dataset ligeramente desbalanceado (más ham).
- Texto corto ideal para TF-IDF.
- Listo para clasificación de spam.

## 3. Preparar datos

In [ ]:
X = df["mensaje"]
y = df["spam"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)

## 4. Modelos con TF-IDF

In [ ]:
modelos = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=RANDOM_STATE),
}

resultados = []
pipes = {}

for nombre, clf in modelos.items():
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=1000)),
        ("model", clf)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    resultados.append({
        "modelo": nombre,
        "accuracy": accuracy_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
    })
    pipes[nombre] = pipe

pd.DataFrame(resultados).sort_values("roc_auc", ascending=False)

## 5. Evaluación detallada

In [ ]:
best = pipes["Logistic Regression"]
y_pred = best.predict(X_test)
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Matriz de confusión")
plt.xlabel("Predicho")
plt.ylabel("Real")
plt.show()

## 6. SHAP (aproximado para texto)

In [ ]:
!pip install -q shap

In [ ]:
import shap

# Usar explainer sobre el modelo lineal
vectorizer = best.named_steps["tfidf"]
model = best.named_steps["model"]

X_train_vec = vectorizer.transform(X_train)
explainer = shap.LinearExplainer(model, X_train_vec)
shap_values = explainer.shap_values(X_train_vec)

print("SHAP listo")

In [ ]:
shap.summary_plot(shap_values, X_train_vec, feature_names=vectorizer.get_feature_names_out(), max_display=10, show=False)
plt.title("SHAP Summary — Spam vs Ham")
plt.tight_layout()
plt.show()

## 7. Conclusiones

- TF-IDF + Logistic Regression funciona muy bien en este ejemplo.
- Palabras clave de spam (premio, clic, gratis) dominan las predicciones.
- Se puede extender a datasets reales de SMS spam.

**Rango esperado (ejemplo):** accuracy > 0.90